[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/coregionalization_toy.ipynb)

# Coregionalized regression (ICM) with `gpc`

GPc port of the spirit of [`GPy/coregionalized_regression_tutorial.ipynb`](../GPy/coregionalized_regression_tutorial.ipynb) / native [coregionalization_toy](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/coregionalization_toy.md).

Requires `gpc>=0.1.0a8` (CIP-0038 Phase D façade; also on `0.1.0a7`).

```python
m = gpc.GPCoregionalizedRegression(X_list, Y_list, kernel=gpc.kern.RBF(1), rank=1)
m.optimize()
mu, var = m.predict_output(Xstar, output=0)
```

**Notes vs GPy**

* Stack via `gpc.util.multioutput.build_XY` / `ICM` / `LCM`.
* Default adds per-output `IndexWhite`; spherical `likelihood.variance` is frozen tiny (CIP-000D / CIP-0027).
* `objective()` is **−log-likelihood**.
* Sparse / MixedNoise multi-output object graph still deferred.


In [ ]:
try:
    import gpc
except ImportError:
    import subprocess
    import sys
    # Prefer PyPI; fall back to TestPyPI for pre-release alphas (GPc CIP-002C).
    cmd = [sys.executable, "-m", "pip", "install", "-q", "gpc"]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-cache-dir",
                "--index-url",
                "https://test.pypi.org/simple/",
                "--extra-index-url",
                "https://pypi.org/simple/",
                "gpc>=0.1.0a8",
            ]
        )
    import gpc


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    import matplotlib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])


## Two noisy sinusoids

Synthesise two outputs of different length (GPy `coregionalization_toy` sizes).

In [ ]:
rng = np.random.default_rng(0)
N0, N1 = 50, 30
X0 = np.linspace(0.0, 8.0, N0)[:, None]
X1 = np.linspace(0.0, 5.0, N1)[:, None]
Y0 = np.sin(X0) + 0.1 * rng.normal(size=X0.shape)
Y1 = np.sin(X1) + 2.0 + 0.1 * rng.normal(size=X1.shape)

## ICM model

Pass a spatial kernel; the façade builds `ICM` and optional `IndexWhite`.

In [ ]:
m = gpc.GPCoregionalizedRegression(
    [X0, X1], [Y0, Y1], kernel=gpc.kern.RBF(1, lengthscale=1.0), rank=1
)
print(m)
print("paths (sample):", m.param_paths()[:8], "...")
m.optimize(iters=40, messages=False)
print("objective", m.objective())

## Predictions per output

In [ ]:
Xs = np.linspace(0.0, 8.0, 120)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, Xi, Yi, out, title in (
    (axes[0], X0, Y0, 0, "output 0"),
    (axes[1], X1, Y1, 1, "output 1"),
):
    mu, var = m.predict_output(Xs, output=out)
    sd = np.sqrt(np.maximum(var, 0.0))
    ax.plot(Xi, Yi, "kx", ms=4, alpha=0.7)
    ax.plot(Xs, mu, "C0-")
    ax.fill_between(
        Xs[:, 0], (mu - 2 * sd)[:, 0], (mu + 2 * sd)[:, 0], color="C0", alpha=0.2
    )
    ax.set_title(title)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
fig.suptitle("ICM + IndexWhite")
fig.tight_layout()
plt.show()

## Explicit ICM / LCM

Helpers mirror GPy `util.multioutput`:

In [ ]:
icm = gpc.util.multioutput.ICM(1, 2, gpc.kern.RBF(1), W_rank=1)
lcm = gpc.util.multioutput.LCM(1, 2, [gpc.kern.RBF(1), gpc.kern.Linear(1)], W_rank=1)
print("ICM:", icm)
print("LCM:", lcm)
m_lcm = gpc.GPCoregionalizedRegression([X0, X1], [Y0, Y1], kernel=lcm, indexwhite=True)
m_lcm.optimize(iters=25)
print("LCM objective", m_lcm.objective())